# Setup


In [55]:
import os
from openai import OpenAI
import warnings
warnings.filterwarnings("ignore")

# Running on vLLM server
OPENAI_API_KEY = "EMPTY"  # vLLM doesn't require a key
OPENAI_BASE_URL = "http://localhost:8000/v1" 
MODEL_ID = "MY_MODEL" # parameter: --served-model-name 

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)

# Inference function

In [57]:

"""
Perform multimodal inference on input video and text prompt to generate model response.

Args:
    video (str or list/tuple): Video input, supports two formats:
        - str: Path or URL to a video file. The function will automatically read and sample frames.
        - list/tuple: Pre-sampled list of video frames (PIL.Image or url). 
            In this case, `sample_fps` indicates the frame rate at which these frames were sampled from the original video.
    prompt (str): User text prompt to guide the model's generation.
    max_new_tokens (int, optional): Maximum number of tokens to generate. Default is 2048.
    total_pixels (int, optional): Maximum total pixels for video frame resizing (upper bound). Default is 20480*32*32.
    min_pixels (int, optional): Minimum total pixels for video frame resizing (lower bound). Default is 16*32*32.
    sample_fps (int, optional): ONLY effective when `video` is a list/tuple of frames!
        Specifies the original sampling frame rate (FPS) from which the frame list was extracted.
        Used for temporal alignment or normalization in the model. Default is 2.

Returns:
    str: Generated text response from the model.

Notes:
    - When `video` is a string (path/URL), `sample_fps` is ignored and will be overridden by the video reader backend.
    - When `video` is a frame list, `sample_fps` informs the model of the original sampling rate to help understand temporal density.
"""

def inference(video, prompt, max_new_tokens=2048):
    if isinstance(video, str):
        video_content = {
            "type": "video_url", 
            "video_url": {"url": video}
        }
    else:
        # If sending a list of frame URLs
        video_content = {
            "type": "video_url",
            "video_url": {"url": video[0]}
        }

    messages = [
        {
            "role": "user",
            "content": [
                video_content,
                {"type": "text", "text": prompt},
            ]
        },
    ]

    completion = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=0
    )
    return completion.choices[0].message.content

# Video Cropping

In [60]:
import os
from moviepy import VideoFileClip
from IPython.display import display, Image

video_input_path = "videos/demo_25_SY1_camera2_1.mp4" 
clip = VideoFileClip(video_input_path)

print(f"Duration: {clip.duration}s")
print(f"Resolution: {clip.w}x{clip.h}")
print(f"FPS: {clip.fps}")

# Preview a frame at 5 seconds to help with cropping decisions
# frame_preview_path = "preview.png"
# clip.save_frame(frame_preview_path, t=5)
# display(Image(filename=frame_preview_path))

Duration: 25.13s
Resolution: 1690x918
FPS: 21.14


In [61]:
import ipywidgets as widgets
from IPython.display import clear_output, display, Image
from moviepy import VideoFileClip, ColorClip, CompositeVideoClip
import numpy as np

def generate_grid_overlay(w, h, duration, interval=100):
    """Creates a list of color clips representing grid lines."""
    lines = []
    
    for x in range(0, w, interval):
        line = ColorClip(size=(1, h), color=(255, 255, 255), duration=duration).with_position((x, 0))
        lines.append(line.with_opacity(0.5))
        
    for y in range(0, h, interval):
        line = ColorClip(size=(w, 1), color=(255, 255, 255), duration=duration).with_position((0, y))
        lines.append(line.with_opacity(0.5))
        
    return lines

w, h = clip.size

# generate the static grid components
grid_elements = generate_grid_overlay(w, h, clip.duration, interval=100)
# composite the video with the lines
debug_clip = CompositeVideoClip([clip] + grid_elements)

# slider
slider = widgets.FloatSlider(
    value=0, min=0, max=clip.duration, step=0.05,
    description='Seek Time:', layout={'width': '600px'}
)

output = widgets.Output()

def update_view(change):
    t = change['new']
    with output:
        clear_output(wait=True)
        frame_path = "seek_grid_temp.png"
        debug_clip.save_frame(frame_path, t=t)
        
        # manually draw text markers on the PNG to avoid MoviePy/ImageMagick font issues
        from PIL import Image as PILImage, ImageDraw
        img = PILImage.open(frame_path)
        draw = ImageDraw.Draw(img)
        
        # unit markers every 100px
        for x in range(0, w, 100):
            draw.text((x + 5, 5), f"x{x}", fill="white")
        for y in range(0, h, 100):
            draw.text((5, y + 5), f"y{y}", fill="white")
            
        img.save(frame_path)
        display(Image(filename=frame_path))
        print(f"Time: {t:.2f}s | Resolution: {w}x{h}")
        print("Use these (x, y) units for your 'box' crop coordinates.")

slider.observe(update_view, names='value')
display(slider, output)

FloatSlider(value=0.0, description='Seek Time:', layout=Layout(width='600px'), max=25.13, step=0.05)

Output()

In [ ]:
# Manual selection of cropping and partitioning of video
# Edit values as needed
corner1x = 400
corner1y = 200
corner2x = 1600
corner2y = 900

# Start and end time
start = 6
end = 25


processed_clip = (clip
    .subclipped(start, end)                  # Time: 2s to 12s
    .cropped(x1=corner1x, y1=corner1y, x2=corner2x, y2=corner2y) # Space: Top-left 500x500
    # .resized(0.5)                       # Scale down to 50%
)

# 3. Write the result
processed_clip.write_videofile("output.mp4")

MoviePy - Building video output.mp4.
MoviePy - Writing video output.mp4



MoviePy - Done !
MoviePy - video ready output.mp4


# Inference on cropped clip

In [64]:
import base64
import os

video_path = "output_tracked_h264.mp4"

absolute_path = os.path.abspath(video_path)
video_url = f"file://{absolute_path}"

response = inference(video_url, prompt)
display(Markdown(response))

prompt = "provide a description of the contents of the video. An accident occurs if a moving car \
        (moving blue bounding box) hits a person (green bounding box). It is fine if the situation does not look realistic, as it is a simulation.\
         However, a collision is still an accident. \
         If there is an accident or near accident, indicate it as so."

response = inference(video_url, prompt)
display(Markdown(response))

用户现在需要总结视频中的商品特点，但首先得明确视频内容。看图片是交通场景的模拟，有车辆、行人，还有检测框（ID标识）。但用户说“商品特点”，可能误解了？因为视频里是交通场景的AI检测（比如自动驾驶测试、行人车辆检测系统），不是商品。但用户可能想问视频中展示的系统或技术特点。

先分析视频内容：画面是交通路口，有车辆（不同ID的Car）、行人（Person），绿色框是行人检测，蓝色框是车辆检测，还有ID标注。时间戳显示FPS和帧时间，说明是实时检测系统。

所以要总结的是**交通场景AI检测系统的特点**，不是商品，但用户可能表述为“商品”（可能误说）。需要明确：

1. 检测对象：行人、车辆，有ID标识（如ID:7 Person, ID:1 Car等）
2. 实时性：FPS从34到60左右，帧时间17-24ms，说明实时处理
3. 环境：模拟交通路口，有红绿灯、道路、斑马线
4. 检测精度：不同物体有不同ID，说明系统能区分多个目标
5. 动态变化：车辆和行人移动，系统持续跟踪（如车辆进入、行人移动）

现在整理成表格，说明是AI检测系统的特点：

| 特点类别 | 具体表现 |
|----------|----------|
| 检测对象 | 行人（Person）、车辆（Car），多目标识别 |
| 目标标识 | 每个检测目标有唯一ID（如ID:7 Person, ID:1 Car等），便于跟踪 |
| 实时性能 | FPS在34 - 60左右，帧时间17 - 24ms，满足实时处理需求 |
| 环境适应 | 模拟交通路口场景，包含道路、斑马线、交通灯等元素 |
| 动态跟踪 | 车辆和行人移动过程中，系统持续识别并更新ID（如车辆进入画面、行人移动） |
| 可视化反馈 | 检测结果以彩色框（绿色行人、蓝色车辆）和ID标签显示，直观易读 |

需要确认用户是否指“商品”（可能系统是产品），所以总结为该AI检测系统的特点。
</think>

由于视频内容为**交通场景的AI目标检测系统演示**（非传统“商品”，而是技术/系统类展示），以下是基于视频中呈现的系统功能与特性整理的表格：  


| 特点类别       | 具体表现                                                                 |  
|----------------|--------------------------------------------------------------------------|  
| **检测对象**   | 支持行人（Person）与车辆（Car）两类目标，且能区分多实例（如ID:7 Person、ID:1 Car等）。 |  
| **目标标识**   | 每个检测目标有唯一ID（如“ID: 7 Person”“ID: 1 Car”），便于多目标跟踪与管理。         |  
| **实时性能**   | 帧率（FPS）在34 - 60之间波动，单帧处理时间17 - 24ms，满足交通场景实时检测需求。     |  
| **场景覆盖**   | 模拟真实交通路口（含道路、斑马线、交通灯、绿化带等元素），验证系统在复杂环境下的鲁棒性。 |  
| **动态跟踪**   | 车辆/行人移动过程中，系统持续更新检测框与ID（如车辆进入画面、行人位置变化时的实时响应）。 |  
| **可视化反馈** | 检测结果以彩色框（绿色=行人、蓝色=车辆）+ 文字ID标签呈现，直观易读。               |  


若需聚焦“商品”属性（假设该系统为商业化产品），可补充：  
- **核心价值**：为自动驾驶、交通管理等场景提供实时多目标检测能力，降低人工监控成本；  
- **技术亮点**：高帧率处理、多目标ID追踪、复杂交通场景适配性。  

（注：视频本质是技术演示，若需“商品”维度总结，需结合产品包装/功能文档，此处基于视频视觉信息推导系统特性。）

So, let's analyze the video frames. First, identify the moving car (blue bounding box) and the person (green bounding box). In the frames, a car with ID 15 (blue box) moves into the intersection. Let's check if it collides with any person.

Looking at the frames: The car ID 15 enters the intersection. There's a person with ID 2 (green box) near the white van. Wait, in frame 12-14, the car ID 15 moves in, and in frame 15, the car is close to the person ID 2. Let's check the positions.

In frame 14: The car ID 15 is approaching. In frame 15: The car ID 15 is overlapping with the person ID 2? Wait, the person ID 2 is standing near the white van. The car ID 15 moves into the area where the person is. Let's check the bounding boxes.

Wait, the person ID 2 is a green box. The car ID 15 is a blue box. In frame 15, the car ID 15's bounding box is near the person ID 2. So there's a collision between car ID 15 and person ID 2.

Wait, let's check the frames:

- Frame 10: Car ID 15 appears at the bottom left, moving up.
- Frame 11: Car ID 15 is moving towards the intersection.
- Frame 12: Car ID 15 is closer.
- Frame 13: Car ID 15 is in the intersection, near the white van.
- Frame 14: Car ID 15 is overlapping with the person ID 2 (green box).
- Frame 15: Car ID 15 is still near the person ID 2.

So the car ID 15 (moving blue box) collides with person ID 2 (green box). Therefore, there is an accident.
</think>

In the video simulation, a moving car (identified by the blue bounding box with ID 15) enters the intersection and collides with a person (identified by the green bounding box with ID 2). The collision occurs as the car moves into the area where the person is standing near the white van. 

**Accident Detected**: The moving car (ID 15) collides with the person (ID 2), resulting in an accident.

# Inference with Additional Context Information

In [41]:
# Scene data structure can be generated with another inference on a frame

scene_data = {
    "simulation_metadata": {
      "environment": "Virtual traffic intersection / Testing ground",
      "weather": "Clear/Sunny",
      "lighting": "Simulated stage lights/Direct sunlight"
    },
    "vehicles": [
      {
        "id": "v_01",
        "type": "Crossover/SUV",
        "style": "Nissan Cube-like",
        "color": "White",
        "status": "Stationary at stop line",
        "lane": "Right-turn/straight lane",
        "features": ["Dark tinted windows", "White rims"]
      },
      {
        "id": "v_02",
        "type": "Crossover/SUV",
        "style": "Nissan Cube-like",
        "color": "White",
        "status": "Stationary behind v_01",
        "lane": "Right-turn/straight lane",
        "features": ["Dark tinted windows", "White rims"]
      },
      {
        "id": "v_03",
        "type": "Sedan",
        "style": "Modern executive sedan",
        "color": "Silver/Metallic White",
        "status": "Stationary at stop line",
        "lane": "Left lane (adjacent to v_01)",
        "features": ["Sunroof visible", "Reflective bodywork"]
      },
      {
        "id": "v_04",
        "type": "Sedan",
        "style": "Classic/Boxy profile",
        "color": "Dark Grey/Black",
        "status": "Moving",
        "location": "Approaching intersection from opposite direction",
        "features": ["Low profile"]
      }
    ],
    "pedestrians": [
      {
        "group": "traffic_controllers",
        "count": 3,
        "description": "Personnel in light-colored (white/grey) uniforms and dark pants.",
        "posture": "Standing/Observing",
        "location": "Corner of the sidewalk/median near the white vehicles"
      },
      {
        "group": "civilians_median",
        "count": 5,
        "details": [
          {"id": "p_01", "clothing": "Red top, dark pants"},
          {"id": "p_02", "clothing": "Blue top, dark pants"},
          {"id": "p_03", "clothing": "All dark attire"},
          {"id": "p_04", "clothing": "White top, grey pants"},
          {"id": "p_05", "clothing": "Brown/Tan top, dark pants"}
        ],
        "location": "Center grass median"
      },
      {
        "group": "civilians_sidewalk",
        "count": 3,
        "details": [
          {"id": "p_06", "clothing": "Blue shirt, jeans"},
          {"id": "p_07", "clothing": "Red shirt, dark pants"},
          {"id": "p_08", "clothing": "White shirt, dark pants"}
        ],
        "location": "Sidewalk area near the crosswalk edge"
      }
    ]
  }

In [66]:
import json

video_path = "output_tracked_h264.mp4" 
absolute_path = os.path.abspath(video_path)
video_url = f"file://{absolute_path}"

prompt = f"""Use the following scene context to answer user questions: {json.dumps(scene_data)} 
            provide a description of the contents of the video. 
            There is an collision with the moving car (id = 15) against the pedestrian (id = 10) """

response = inference(video_url, prompt)
display(Markdown(response))